# LFPC Policy-Parameter Sensitivity Analysis v2

This notebook rebuilds the sensitivity section from the latest experiment registry. It separates exact contexts and measures policy stability, not only metric movement.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

REPORT_DIR = Path('report_artifacts/policy_parameter_sensitivity_report')
TABLE_DIR = REPORT_DIR / 'tables'
PLOT_DIR = REPORT_DIR / 'plots'

def read_table(name):
    path = TABLE_DIR / name
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

tables = {
    'Context policy stability': read_table('policy_stability_by_context.csv'),
    'Layerwise policy stability': read_table('layerwise_policy_stability.csv'),
    'Threshold response': read_table('threshold_response_by_context.csv'),
    'Parameter sensitivity curves': read_table('parameter_sensitivity_curves.csv'),
    'Pooled parameter effects': read_table('pooled_parameter_effects.csv'),
    'Pooled standardized parameter effects': read_table('pooled_standardized_parameter_effects.csv'),
    'Report manifest': read_table('plot_manifest.csv'),
}
for title, df in tables.items():
    display(Markdown(f'## {title}'))
    display(df.head(80) if not df.empty else Markdown('No rows available.'))


In [ ]:
display(Markdown('## Sensitivity Plots'))
manifest = tables['Report manifest']
for _, row in manifest.iterrows():
    path = Path(row['plot'])
    if path.exists():
        display(Markdown(f"### {row.get('description', path.name)}"))
        display(Image(filename=str(path)))


In [ ]:
display(Markdown('## Least Stable Context Drill-Down'))
summary = tables['Context policy stability']
layer = tables['Layerwise policy stability']
if not summary.empty:
    cols = ['objective_label','dataset','model','scope','ratio','num_threshold_settings','modal_policy_share','mean_layer_dominant_share','accuracy_delta_range','flops_reduction_range','time_sec_range','best_stack_id','best_methods_used']
    display(summary[[c for c in cols if c in summary.columns]].sort_values(['modal_policy_share','mean_layer_dominant_share']).head(20))
if not layer.empty:
    cols = ['objective_label','dataset','model','scope','ratio','layer','dominant_method_display','dominant_method_share','num_methods_seen','method_frequency_json']
    display(layer[[c for c in cols if c in layer.columns]].sort_values(['dominant_method_share','num_methods_seen'], ascending=[True, False]).head(80))


## Regression Model Used

For the unstandardized pooled model, each response is fitted as:

$$y_i = \beta_0 + \beta_1 v_i + \beta_2 s_i + \beta_3 j_i + \beta_4 r_i + \gamma_o O_i + \gamma_d D_i + \gamma_m M_i + \gamma_q Q_i + \epsilon_i$$

where $v_i$ is the variance threshold, $s_i$ the Spearman threshold, $j_i$ the Jaccard threshold, $r_i$ the prune ratio, and the $\gamma$ terms are fixed-effect controls for objective, dataset, model, and scope.

For the standardized coefficient plot/table, the same threshold predictors are z-scored and fitted as:

$$z(y_i) = \beta_0 + \beta_1 z(v_i) + \beta_2 z(s_i) + \beta_3 z(j_i) + \beta_4 z(r_i) + \epsilon_i$$

The standardized model is intended for effect-size comparison; the fixed-effect pooled model is intended for adjusted inference.
